# trainer-subclass-extend — ex1: subclass a trainer and extend _step via super() delegation

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `trainer-subclass-extend`. Running the final beacon cell reports progress against the `Trainer: subclass extend pattern` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: subclass extend pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-subclass-extend`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-subclass-extend"
DD_SUBTOPIC = "Trainer: subclass extend pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Trainer: subclass-extend pattern — quick refresher

When you want to ADD behavior to a base trainer's `_step` (per-batch hook) without rewriting it, override + `super()`:

```python
class BaseTrainer:
    def _step(self, batch):
        x, y = batch
        out = self.model(x)
        loss = self.loss_fn(out, y)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        return {'loss': loss.item()}

class MyTrainer(BaseTrainer):
    def _step(self, batch):
        out = super()._step(batch)        # run the base step first
        out['my_extra_metric'] = self._compute_extra(batch)  # add to it
        return out
```

**Three rules of the extend pattern.**
1. Call `super()._step(batch)` FIRST so the base does its work.
2. ADD to the result (don't replace it) — extend the dict.
3. RETURN the extended result — callers expect the same shape the base returned, plus your additions.

**Why not write a new `_step` from scratch.** Duplicating the base body works, but every fix to `BaseTrainer._step` would need to be re-applied to your subclass. The `super()._step()` call inherits ALL future fixes for free.

**The 'before / after' variations.**
- BEFORE: `self._pre_hook(); out = super()._step(batch); return out` — run a hook, then delegate.
- AFTER:  `out = super()._step(batch); self._post_hook(out); return out` — delegate, then a hook.
- AROUND: do both — the most common shape for logging.

**Compose, don't replace.** If you find yourself NOT calling `super()._step()` in an override, you're not extending — you're replacing. Make a new class instead so the inheritance tells the right story.

### Exercise 1 — subclass a trainer and extend _step via super() delegation

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the subclass-extend pattern: override `BaseTrainer._step` in a subclass that calls `super()._step(batch)` first and then ADDS a new metric to the returned dict.
> Keywords: subclass, super, hook, trainer
> ```

**KCs targeted:** `super-step-delegate-then-extend`, `preserve-base-return-shape`

`BaseTrainer` is provided in the stub. It has a `_step(batch)` method that runs the base training step and returns a dict like `{'loss': <float>}`.

Implement `LoggingTrainer(BaseTrainer)`. In its `_step(batch)`:
1. Call `super()._step(batch)` FIRST. Capture the result (a dict).
2. Compute `extra = batch[0].abs().mean().item()` — the mean absolute value of the input tensor. This is a fake 'input magnitude' metric used to demonstrate the extend pattern.
3. Add `'input_mag': extra` to the dict.
4. Return the EXTENDED dict — must contain BOTH the base's `'loss'` key AND your new `'input_mag'` key.

Constraints:
- MUST call `super()._step(batch)` — do not duplicate the base body.
- MUST return the same dict object (or a dict with the same `loss` value) — base callers expect the loss key intact.
- Do NOT add `__init__`; inherit it from `BaseTrainer`.

In [ ]:
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, lr=1e-2):
        self.model = model
        self.opt = t.optim.SGD(model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def _step(self, batch):
        x, y = batch
        pred = self.model(x)
        loss = self.loss_fn(pred, y)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return {'loss': loss.item()}

class LoggingTrainer(BaseTrainer):
    def _step(self, batch):
        # 1. Delegate to the base step (runs the actual training step,
        #    returns {'loss': <float>}).
        out = super()._step(batch)
        # 2. Extend with an extra metric.
        out['input_mag'] = batch[0].abs().mean().item()
        # 3. Return the extended dict (still contains base's 'loss').
        return out


<details><summary>Solution</summary>

```python
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, lr=1e-2):
        self.model = model
        self.opt = t.optim.SGD(model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def _step(self, batch):
        x, y = batch
        pred = self.model(x)
        loss = self.loss_fn(pred, y)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return {'loss': loss.item()}

class LoggingTrainer(BaseTrainer):
    def _step(self, batch):
        # 1. Delegate to the base step (runs the actual training step,
        #    returns {'loss': <float>}).
        out = super()._step(batch)
        # 2. Extend with an extra metric.
        out['input_mag'] = batch[0].abs().mean().item()
        # 3. Return the extended dict (still contains base's 'loss').
        return out
```

**Three lines, full extend pattern.** `super()._step(batch)` does the heavy lifting. The subclass only adds what's new — the input-magnitude metric — and returns the extended dict.

**Why not write a fresh `_step` from scratch.** You'd duplicate the optimizer step, the zero_grad, the backward — and any future fix to `BaseTrainer._step` (e.g. adding gradient clipping) would need to be re-applied to your subclass. With `super()._step()`, the fix is inherited for free.

**The 'before/after/around' variants.** BEFORE: `self._pre(); out = super()._step(batch); return out`. AFTER: `out = super()._step(batch); self._post(out); return out`. AROUND: do both — most common for logging hooks.

**Inherit `__init__` for free.** Because `LoggingTrainer` doesn't define its own `__init__`, it uses `BaseTrainer.__init__`. If you needed extra subclass-specific state, you'd write `def __init__(self, model, lr=1e-2, log_dir=None): super().__init__(model, lr); self.log_dir = log_dir`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()